<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c05-header.png" alt="Nextia Learning · Machine Learning: From Problem to Reliable Model" width="100%">

# Supervised learning

**[Supervised learning](https://learning.nextia-ai.com/courses/ml/m01/supervised-learning/)** · Machine Learning: From Problem to Reliable Model · Module 1, lesson 2 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** train a model from examples that have the right answer (a label), use it to predict the label of new examples, and split a table into its features `X` and its label `y`.

| | |
|---|---|
| **Time** | About 25 minutes |
| **Needs** | An internet connection for the setup cells. No account. The first cell installs pandas 3.0.6 and scikit-learn 1.9.1. |
| **You should know** | Training, model and prediction, from [What machine learning is](https://learning.nextia-ai.com/courses/ml/m01/what-machine-learning-is/). |
| **You do not need** | A local project. The setup below downloads the data. |
| **Tested** | 2026-10-07, Python 3.14.6 with pandas 3.0.6, scikit-learn 1.9.1 and matplotlib 3.11.2, in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/ml/m01/supervised-learning/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C05/M01-L02-supervised-learning/supervised-learning-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs pandas 3.0.6 and scikit-learn 1.9.1, the versions that the course uses, and matplotlib for the charts. It needs an internet connection and takes up to a minute the first time.

You should see `pandas 3.0.6` and `scikit-learn 1.9.1`. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.0.6 scikit-learn==1.9.1 matplotlib
import pandas; print("pandas", pandas.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)

## Setup: get the practice data

Run the next cell. It downloads the practice data of this course (about 2.5 MB) into a folder `ticket-model/data`, and checks each file's checksum. It is the same data that `get_data.py` downloads in [Define a baseline](https://learning.nextia-ai.com/courses/ml/m02/define-a-baseline/). It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: train 16939 rows (1983 escalated), valid 1179 (183), test 1002 (157).` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings (it needs a verified phone number).
- `wrong checksum`: delete the folder `ticket-model` (in Colab: the **Files** panel on the left), then run the cell again.

The data is made up for this course; it has no real people.

In [ ]:
# Setup: download the C05 data into ticket-model/data/ and check it.
# Standard library only: the same steps as get_data.py.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C05/"
FILES = {
    "train.csv": "63b8732a28e891189cc28a9bcd505a093a9c4c51c01f3d0e1fba6d4959242d53",
    "valid.csv": "3604df83ea2ec3cec70c2eedfc7e95cc6d5cb82ccfc3b5cf565e614783e2bde2",
    "test.csv": "60e89f75e4d7d59ccfbda49ca719ce7f8e8d97c0380256d633ab240d546771ec",
    "extra_features.csv": "03c6da28bcb347982aa70a7c72b869771a6cb055654d629da924512c6fd56563",
    "daily_tickets.csv": "7deba76b112b4133a5d16b74be7e2b331ce28c7064e7fd8dd471768e5f1ba360",
    "july_tickets.csv": "10dffc093993341a361172d0aa6ea0e7c06521f140cc2a80ee90c713361f7c99",
    "july_labels.csv": "3163c57e398db6fd1c19e505548d0fa1464821ef96c77901a759abbd3a8605fa",
}
PROJECT_FILES = []
HERE = Path.cwd()
# The notebook works in its own folder ticket-model/, marked with a file, so
# it never writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("ticket-model").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected=None):
    """Use a copy next to the notebook if there is one, or download it."""
    if path.exists() and (expected is None or sha256(path) == expected):
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if expected and sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder ticket-model and run again.")

for name, expected in FILES.items():
    fetch(name, LABS + "data/" + name, PROJECT / "data" / name, expected)
for name in PROJECT_FILES:
    fetch(name, LABS + "project/" + name, PROJECT / name)

os.chdir(PROJECT)  # work in the project folder, as in the lesson
counts = {}
for part in ("train", "valid", "test"):
    rows = Path(f"data/{part}.csv").read_text(encoding="utf-8").splitlines()[1:]
    counts[part] = (len(rows), sum(row.endswith(",1") for row in rows))
print("Ready: train {} rows ({} escalated), valid {} ({}), test {} ({}).".format(
    *counts["train"], *counts["valid"], *counts["test"]))

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

It also loads pandas as `pd` and numpy as `np`, and sets pandas to show 4 decimals. Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it.
import hashlib, json, warnings
import numpy as np
import pandas as pd
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=FutureWarning)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The colour examples

A child learns colour names from examples: you show a red ball and say "red", a blue cup and say "blue". The [lesson page](https://learning.nextia-ai.com/courses/ml/m01/supervised-learning/#learning-colours-with-a-child) tells the full story.

A computer sees a colour as three numbers: the amount of red, green and blue light, each from 0 to 255. Run the next cell. It makes 15 colour swatches, three for each name. Each row is one **example**. The three numbers are the **features**: what the learner looks at. The name is the **label**: the right answer that a teacher gives.

In [ ]:
swatches = pd.DataFrame(
    [("red", 220, 40, 40), ("red", 190, 25, 35), ("red", 235, 70, 60),
     ("orange", 245, 135, 35), ("orange", 235, 115, 25), ("orange", 250, 155, 60),
     ("yellow", 250, 220, 50), ("yellow", 240, 205, 40), ("yellow", 255, 235, 100),
     ("green", 50, 165, 70), ("green", 90, 190, 80), ("green", 30, 130, 55),
     ("blue", 40, 100, 200), ("blue", 80, 140, 225), ("blue", 30, 70, 170)],
    columns=["name", "red", "green", "blue"],
)
X = swatches[["red", "green", "blue"]]  # the features
y = swatches["name"]                    # the label
print(X.shape, y.shape)
swatches.head(4)

> **Check.** You should see `(15, 3) (15,)`, then the first four swatches: three reds and one orange.

The features go in a table called `X`, with one row per example. The labels go in a column called `y`, with one value per example. This is the shape that every scikit-learn model expects.

Run the next cell to see the swatches as colours, with their labels.

In [ ]:
import matplotlib.pyplot as plt


def show_swatches(table, names, title):
    """Draw each row of table (red, green, blue) as a square, with a name under it."""
    fig, axes = plt.subplots(1, len(table), figsize=(1.0 * len(table), 1.4))
    for ax, (_, row), name in zip(np.atleast_1d(axes), table.iterrows(), names):
        ax.imshow([[row[["red", "green", "blue"]].to_numpy(dtype=float) / 255]])
        ax.set_xticks([]); ax.set_yticks([]); ax.set_xlabel(name)
    fig.suptitle(title)
    plt.show()


show_swatches(X, y, "15 examples, each with its label")

> **Check.** You should see 15 coloured squares, each with its name under it.

## 2. Train a model, then name new colours

The model is a **nearest-neighbour classifier**. For a new swatch, it looks for the 3 examples that are most similar (the nearest numbers), and it gives their most common name. That is all you need to know about it now.

Training is one line: `fit(X, y)`. The model learns from the features **and** the labels.

> **Predict.** The cell names three new swatches that the model never saw: (200, 60, 50), (110, 170, 220) and (240, 125, 40). Look at the numbers. Which names do you expect? Then run the cell.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

model = KNeighborsClassifier(n_neighbors=3)
model.fit(X, y)  # training: learn from examples with labels

new = pd.DataFrame([(200, 60, 50), (110, 170, 220), (240, 125, 40)], columns=["red", "green", "blue"])
predictions = model.predict(new)  # prediction: no labels here
print(list(predictions))
show_swatches(new, predictions, "Three new swatches and the predicted names")

> **Check.** You should see `['red', 'blue', 'orange']`, then the three squares with these names.

The model never saw these three swatches, and it named all three as a person would. Doing well on new examples is called **generalisation**. It is the only thing that matters: a model that knows only its examples is useless.

Run the next cell to see why the model chose "orange" for the third swatch: its 3 nearest examples, and how far each one is.

In [ ]:
distances, rows = model.kneighbors(new.iloc[[2]])
nearest = swatches.iloc[rows[0]].assign(distance=distances[0].round(0))
nearest

> **Check.** You should see three orange swatches, at distances `12`, `19` and `37`.

## 3. Your turn: name two more swatches

> **Your turn.** Put the two swatches (60, 140, 90) and (245, 215, 90) in a DataFrame `practice`, with the columns `red`, `green` and `blue`. Then put the model's names for them in `practice_names` (as a list). Predict the names before you run the cell.

In [ ]:
practice = ...        # a DataFrame with two rows and the columns red, green, blue
practice_names = ...  # list(model.predict(...))
print(practice_names)

In [ ]:
expect_hash('practice_names', practice_names, '2ad4712744f32ca1')

## 4. When the label is a number: regression

The colour name is a **class**: one of a few names. Predicting a class is **classification**. Many questions need a number instead: a price, a number of minutes, a number of tickets. Predicting a number is **regression**.

The idea is the same: learn from examples with the right answer. Run the next cell. It has 8 flats, each with its size in square metres (the feature) and its price in thousands of dollars (the label). The model predicts the price of a flat of 70 square metres from the 3 most similar flats: it takes the mean of their prices.

> **Predict.** The flats of 65 and 80 square metres cost 255 and 300. What price do you expect for 70 square metres?

In [ ]:
from sklearn.neighbors import KNeighborsRegressor

flats = pd.DataFrame({
    "size_m2": [35, 42, 50, 58, 65, 80, 95, 110],
    "price_k": [150, 175, 200, 230, 255, 300, 345, 390],
})
price_model = KNeighborsRegressor(n_neighbors=3)
price_model.fit(flats[["size_m2"]], flats["price_k"])
new_flat = pd.DataFrame({"size_m2": [70]})
print(round(price_model.predict(new_flat)[0], 2))

> **Check.** You should see `261.67`: the mean of 255, 300 and 230 (the flats of 65, 80 and 58 square metres).

For a price, an answer of 262 when the truth is 265 is nearly right. For a colour name, "orange" when the truth is "red" is wrong. That is the practical difference between regression and classification. Run the next cell to see the flats and the prediction.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.scatter(flats["size_m2"], flats["price_k"], label="examples (size and price)")
ax.scatter([70], price_model.predict(new_flat), marker="x", s=90, label="prediction for 70 m²")
ax.set_xlabel("Size (square metres)")
ax.set_ylabel("Price (thousand $)")
ax.legend()
plt.show()

> **Check.** You should see 8 dots that rise from left to right, and a cross at 70 square metres, between the dots of 65 and 80.

> **Your turn.** Put the predicted price of a flat of 100 square metres, rounded to 2 decimals, in `price_100`. Before you run it, say which 3 flats the model uses.

In [ ]:
price_100 = ...  # round(price_model.predict(...)[0], 2)
print(price_100)

In [ ]:
expect_hash('price_100', price_100, '1989be43a442a132')

## 5. Larkfield: features and label of a ticket

Larkfield's history has the same shape. One row is one ticket. Ten columns describe the ticket when it is created: these are the **features**. The column `escalated_72h` says what happened later: this is the **label**. Run the next cell. It reads the train tickets and shows three of them.

In [ ]:
train = pd.read_csv("data/train.csv", dtype={"customer_id": "string"},
                    keep_default_na=False, na_values=[""])
FEATURES = ["channel", "team", "segment", "region", "priority", "order_value", "word_count",
            "customer_tenure_days", "prior_tickets_90d", "created_hour"]
TARGET = "escalated_72h"
train[FEATURES + [TARGET]].head(3)

> **Check.** You should see three tickets: `delivery`, `warranty` and `returns`, each with `escalated_72h` equal to 0.

Run the next cell. It splits the table into `X` and `y`, as for the colours.

In [ ]:
X = train[FEATURES]
y = train[TARGET]
print("X:", X.shape, " y:", y.shape)
print(y.value_counts())

> **Check.** You should see `X: (16939, 10)  y: (16939,)`, then `14956` tickets with 0 and `1983` with 1.

The label has two classes, 0 and 1, so predicting it is classification. The label of each past ticket comes from a later event: a senior agent took the ticket, or did not, within 72 hours. Nobody had to label the tickets by hand. That is a lucky case: labels often cost time and money.

## 6. Change one thing: a number as the label

> **Your turn.** The file `data/daily_tickets.csv` has one row per day. Read it into `daily`. Make `X_daily` with the features `weekday`, `is_weekend` and `month`, and `y_daily` with the label `tickets`. Is this classification or regression?

In [ ]:
daily = ...    # read data/daily_tickets.csv
X_daily = ...  # three feature columns
y_daily = ...  # the label column

In [ ]:
expect_hash('(X_daily.shape, y_daily.shape)', (X_daily.shape, y_daily.shape) if hasattr(X_daily, "shape") and hasattr(y_daily, "shape") else None, 'ded20be2434ba9eb')

The label `tickets` is a number of tickets per day, so predicting it is regression. You train this model in Module 4.

## 7. Failure case: wrong labels

A model learns what its labels say, even when they are wrong. Say a helper labelled the swatches, and called two of the three orange swatches "red". Run the next cell. It trains the same model on these labels.

> **Predict.** What name will the new model give the orange swatch (240, 125, 40)?

In [ ]:
wrong_y = swatches["name"].copy()
wrong_y.iloc[[3, 4]] = "red"  # two orange swatches with the wrong label
wrong_model = KNeighborsClassifier(n_neighbors=3).fit(swatches[["red", "green", "blue"]], wrong_y)
print("New orange swatch:", wrong_model.predict(new.iloc[[2]])[0])
agree = (wrong_model.predict(swatches[["red", "green", "blue"]]) == wrong_y).sum()
print("The model agrees with", agree, "of its 15 labels")

> **Check.** You should see `New orange swatch: red` and `The model agrees with 15 of its 15 labels`.

The model agrees with every label it was given, so nothing looks wrong. But it now calls orange "red", because the supervisor taught it so. A model is only as good as its labels. Check a sample of the labels by hand before you train, and ask how each label was made.

## Check your understanding and recap

Answer the **knowledge checks** on the [lesson page](https://learning.nextia-ai.com/courses/ml/m01/supervised-learning/#knowledge-check). They count toward your certificate when you are signed in and enrolled.

In this notebook, a model learned colour names from 15 labelled examples and named new swatches correctly: classification. The same idea predicted the price of a flat from its size: regression. Larkfield's tickets have 10 features (`X`) and one label (`y`, `escalated_72h`). Learning from examples with labels is **supervised learning**: the labels act as the supervisor. And wrong labels teach the wrong answer, without any warning.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Unsupervised learning](https://learning.nextia-ai.com/courses/ml/m01/unsupervised-learning/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/ml/m01/supervised-learning/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.